In [ ]:
import polars as pl
from rdkit import Chem
from itertools import chain

In [14]:
def extract_unique_smiles(df: pl.DataFrame, req_ids: list[str] = []) -> set[str]:
    if req_ids:
        df = df.filter(pl.col("id").is_in(req_ids))
    smarts = df['smarts'].to_list()
    mols = set(
        chain(
            *[
                side.split('.') for sma in smarts for side in sma.split('>>')
            ]
        )
    )
    return mols

In [15]:
fp = '../artifacts/coa_mutase_paths/250912_3hpa_paths.csv'
df = pl.read_csv(fp)

In [16]:
pmols = extract_unique_smiles(df)

In [18]:
krfp = "../artifacts/known/known_reactions.parquet"
krs = pl.read_parquet(krfp)
krs.head()

id,smarts,enzymes,reverse,db_ids
str,str,list[str],str,list[str]
"""e908a824c912d1e39c46de92d1f738…","""**.NC(CCC(=O)NC(CS)C(=O)NCC(=O…","[""P30109"", ""P57108"", … ""P46436""]","""3bbe8e6dca0da1c745e7678c8efebe…","[""RHEA:16438""]"
"""d86d99a8143d3be8fc861a5de5e625…","""*.*.*.*.*.*.*.*.*.*.CC(C)(COP(…","[""Q0UK48"", ""A0A0C6E0I7"", … ""A0A0C6DWS6""]","""e99fb0b75e799eba72aee21c1d04ee…","[""RHEA:51350""]"
"""2972b2db66715ba6f3d6aeaf07f31b…","""*.*.*.*.*.*.*.*.*NC(COP(=O)(O)…","[""G0REX6"", ""P0DO30"", ""A0A482N9V7""]","""5a7fffef224c65b7cd431ddc668eb2…","[""RHEA:64546""]"
"""572e4a84a946af07c9ab6fd1c0347b…","""*.*.*.*.*.*.*.*.*OP(=O)(O)OCC(…","[""Q6M083"", ""Q8U4J0"", … ""C5A6E5""]","""c4c936d8d985a457a128601b8d2769…","[""RHEA:64377""]"
"""dfe7fc761de3c213bf2b18fae4f8a5…","""*.*.*.*.*.*.*.*.CC(C)CCCC(C)CC…","[""Q6M083"", ""Q8U4J0"", … ""C5A6E5""]","""2e026380a550baa6b343604e6f5500…","[""RHEA:64369""]"


In [27]:
req_krids = [
    'd7eea5a1a938257639a2cbe5950fa3501a3fc821',
    'b4c25c288c30a8a0c671f4f171296e4f6785c7ba',
    'e28526a64f436e83d38207b68585369173b63e35',
    '4ef133095b54597cf464d3640d65a15cb75ba461',
    '7fa14c20e24cde2f3097739e7be1e2d30ddda6ef',

]

In [28]:
kmols = extract_unique_smiles(krs, req_krids)

In [29]:
with open("../artifacts/coa_mutase_paths/unique_smiles_for_chemdraw.txt", "w") as f:
    for smi in pmols | kmols:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            f.write(f"{smi}\n")

In [30]:
krs.filter(pl.col("id").is_in(req_krids)).select(["id", "enzymes", "db_ids"])

id,enzymes,db_ids
str,list[str],list[str]
"""4ef133095b54597cf464d3640d65a1…","[""A9X6P9"", ""P76518""]","[""RHEA:37884""]"
"""d7eea5a1a938257639a2cbe5950fa3…","[""Q8VCH6"", ""Q15392"", … ""Q60HC5""]","[""RHEA:36393""]"
"""b4c25c288c30a8a0c671f4f171296e…","[""Q5KUG0"", ""A7IQE6"", … ""A7IQE5""]","[""RHEA:52621""]"
"""e28526a64f436e83d38207b6858536…","[""O51628"", ""P13702""]","[""RHEA:14835""]"
"""7fa14c20e24cde2f3097739e7be1e2…",[],"[""RHEA:24761""]"
